In [1]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import harmonypy as hm
import scanorama

script_loc = Path('01_load_and_preprocess.py').resolve().parent
data_loc = script_loc.parent / "data"
resources_loc = script_loc.parent / 'resources'

In [ ]:
#Read in preprocessed data and copy
scdata = sc.read_h5ad(data_loc / 'intermediate/c_preintegration.h5ad')
hcdata = scdata.copy()

In [ ]:
#Integrate data using harmonypy
pcs = hcdata.obsm['X_pca']
harmony_out = hm.run_harmony(pcs, hcdata.obs, "id")
hcdata.obsm['X_pca_harmony'] = harmony_out.Z_corr

MemoryError: Unable to allocate 782. MiB for an array with shape (205032528,) and data type int32

In [ ]:
#Calculate neighbors and umap on corrected pcs
sc.pp.neighbors(hcdata, use_rep='X_pca_harmony')
sc.tl.umap(hcdata)

In [ ]:
#Integrate data using scanorama

#Make list of batches
scdata['raw_counts'] = scdata.X.copy()
scdata.X = scdata.layers['CP10k_counts']
batch_key = 'id'
mdatas = [scdata[scdata.obs[batch_key] == batch_value].copy() for batch_value in scdata.obs[batch_key].unique()]

#Run scanorama and collect correct space
scanorama.integrate_scanpy(mdatas, dimred = 100)
scanorama_int = [ad.obsm['X_scanorama'] for ad in mdatas]

all_s = np.concatenate(scanorama_int)

scdata.obsm['X_scanorama'] = all_s

In [ ]:
#Calculate neighbors and umap on corrected pcs
sc.pp.neighbors(scdata, use_rep='X_scanorama')
sc.tl.umap(scdata)